# Phần 1: Cài Đặt OLS và Các Hàm Liên Quan

Notebook này trình bày các hàm cài đặt từ đầu theo yêu cầu của Phần 1 trong đề bài.

## Danh sách các hàm cài đặt:
1. `ols_fit(X, y)` — Tính β = (X'X)⁻¹X'y và σ²
2. `hat_matrix(X)` — Tính H = X(X'X)⁻¹X', kiểm tra idempotent
3. `model_metrics(y, y_hat, p)` — Tính RSS, TSS, R², R̄², kiểm định F
4. `coef_inference(X, y, beta_hat, sigma2)` — Tính standard errors, t-statistics, p-values, khoảng tin cậy 95%
5. `vif(X)` — Tính VIF cho từng biến
6. `ridge_fit(X, y, lam)` — Cài đặt Ridge Regression, về ridge trace
7. `residual_plots(X, y, beta_hat)` — Về 4 biểu đồ phân tích phần dư
8. `kfold_cv(X, y, k)` — Cài đặt k-fold cross-validation, tính CV score
9. `simulate_gauss_markov(X, beta_true, sigma, n_simulations)` — Mô phỏng Monte Carlo để kiểm chứng E[β̂] = β và OLS có phương sai nhỏ nhất

## Setup

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add part1 to path for imports
sys.path.insert(0, str(Path.cwd()))

# Import all functions from part1 package
from ols_implementation import ols_fit, hat_matrix, run_ols_analysis
from model_evaluation import model_metrics, residual_plots
from inference import coef_inference, vif
from regularization import ridge_fit, ridge_trace
from cross_validation import kfold_cv, kfold_cv_ridge
from gauss_markov_sim import simulate_gauss_markov, verify_assumptions

# Set random seed for reproducibility
np.random.seed(42)

print("Các modules đã được import thành công!")

## 1. Tạo dữ liệu mẫu

In [ ]:
# Create synthetic data for demonstration
np.random.seed(42)
n_obs, n_feat = 100, 3

# True parameters
beta_true = np.array([2.0, 1.5, -0.8, 0.5])

# Design matrix with intercept column
X_raw = np.random.randn(n_obs, n_feat)
X = np.column_stack([np.ones(n_obs), X_raw])

# Generate response with noise
sigma_true = 0.5
eps = np.random.randn(n_obs) * sigma_true
y = X @ beta_true + eps

# Convert to lists for our pure Python implementation
X_list = X.tolist()
y_list = y.tolist()

print(f"Data shape: X {X.shape}, y {y.shape}")
print(f"True parameters: β = {beta_true}")
print(f"True noise variance: σ² = {sigma_true**2:.4f}")

## 2. OLS Fitting (ols_fit + hat_matrix)

In [ ]:
# Fit OLS model
results = run_ols_analysis(X_list, y_list)
ols_res = results['ols']
hat_res = results['hat']

print("\n" + "="*60)
print("OLS FITTING RESULTS")
print("="*60)
print(f"Status: {ols_res.message}")
print(f"\nEstimated coefficients (β̂):")
for j, b in enumerate(ols_res.beta_hat):
    print(f"  β̂_{j} = {b:10.6f}  (true: {beta_true[j]:10.6f})")
print(f"\nEstimated σ² = {ols_res.sigma2_hat:.6f}  (true: {sigma_true**2:.6f})")
print(f"RSS = {ols_res.rss:.6f}")
print(f"DoF = {ols_res.dof}")

print("\n" + "="*60)
print("HAT MATRIX PROPERTIES")
print("="*60)
print(hat_res.message)

## 3. Verification against NumPy

In [ ]:
# Verify OLS estimates against NumPy
beta_numpy, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
diff_beta = np.max(np.abs(np.array(ols_res.beta_hat) - beta_numpy))

print(f"\nVerification against NumPy:")
print(f"  ||β̂_ours - β̂_numpy||_∞ = {diff_beta:.2e}")
print(f"  Result: {'✓ PASSED' if diff_beta < 1e-8 else '✗ FAILED'}")

# Verify Hat Matrix
H_numpy = X @ np.linalg.inv(X.T @ X) @ X.T
diff_H = np.max(np.abs(np.array(hat_res.H) - H_numpy))
print(f"\n  ||H_ours - H_numpy||_∞ = {diff_H:.2e}")
print(f"  Result: {'✓ PASSED' if diff_H < 1e-8 else '✗ FAILED'}")

## 4. Model Evaluation Metrics

In [ ]:
# Calculate model metrics
y_hat = ols_res.y_hat
p = len(ols_res.beta_hat) - 1
metrics = model_metrics(y_list, y_hat, p)

print("\n" + "="*60)
print("MODEL EVALUATION METRICS")
print("="*60)
print(f"n = {metrics.n}, p = {metrics.p}")
print(f"\nRSS  (Residual Sum of Squares) = {metrics.rss:.6f}")
print(f"TSS  (Total Sum of Squares)     = {metrics.tss:.6f}")
print(f"ESS  (Explained Sum of Squares) = {metrics.ess:.6f}")
print(f"\nR² = {metrics.r2:.6f}")
print(f"R̄² (Adjusted R²) = {metrics.r2_adj:.6f}")
print(f"RMSE = {metrics.rmse:.6f}")
print(f"F-statistic = {metrics.f_statistic:.6f}")

## 5. Coefficient Inference (t-tests, p-values, CI)

In [ ]:
# Calculate coefficient inference
inference = coef_inference(X_list, y_list, ols_res.beta_hat, ols_res.sigma2_hat, alpha=0.05)

print("\n" + "="*60)
print("COEFFICIENT INFERENCE (α = 0.05)")
print("="*60)
print(f"\n{'Var':<8} {'β̂':<12} {'SE(β̂)':<12} {'t-stat':<12} {'p-value':<12} {'95% CI':<25}")
print("-" * 85)

for j in range(len(inference.coefficients)):
    beta_j = inference.coefficients[j]
    se_j = inference.std_errors[j]
    t_j = inference.t_statistics[j]
    p_j = inference.p_values[j]
    ci_l = inference.ci_lower[j]
    ci_u = inference.ci_upper[j]
    
    var_name = f"β̂_{j}" if j > 0 else "Intercept"
    print(f"{var_name:<8} {beta_j:12.6f} {se_j:12.6f} {t_j:12.6f} {p_j:12.6f} [{ci_l:8.4f}, {ci_u:8.4f}]")

## 6. Variance Inflation Factors (VIF)

In [ ]:
# Calculate VIF
vif_result = vif(X_list, threshold=10.0)

print("\n" + "="*60)
print("VARIANCE INFLATION FACTORS (VIF)")
print("="*60)
print(f"\nThreshold for multicollinearity: {10.0}")
print(f"\n{'Variable':<12} {'VIF Value':<15} {'Status':<20}")
print("-" * 50)

for j, (name, vif_val) in enumerate(zip(vif_result.column_names, vif_result.vif_values)):
    status = "⚠️ High multicollinearity" if vif_val > 10 else "✓ OK"
    if vif_val == float('inf'):
        print(f"{name:<12} {'∞':<15} {status:<20}")
    else:
        print(f"{name:<12} {vif_val:>14.4f} {status:<20}")

print(f"\nMax VIF = {vif_result.max_vif:.4f}")
print(f"Multicollinearity: {'Detected ✗' if vif_result.has_multicollinearity else 'None detected ✓'}")

## 7. Ridge Regression

In [ ]:
# Compare OLS with Ridge regression
ridge_results = {}
lambdas = [0, 0.1, 0.5, 1.0, 5.0]

print("\n" + "="*80)
print("RIDGE REGRESSION COMPARISON")
print("="*80)

for lam in lambdas:
    ridge_res = ridge_fit(X_list, y_list, lam)
    ridge_results[lam] = ridge_res
    print(f"\nλ = {lam:<5.1f}: {ridge_res.message}")

# Print coefficients comparison
print(f"\n{'λ':<8} {'β̂₁':<12} {'β̂₂':<12} {'β̂₃':<12} {'β̂₄':<12}")
print("-" * 60)
for lam in lambdas:
    res = ridge_results[lam]
    coefs = [f"{b:12.6f}" for b in res.coefficients]
    print(f"{lam:<8.1f} {' '.join(coefs)}")

## 8. Residual Analysis

In [ ]:
# Residual analysis
residual_data = residual_plots(X_list, y_list, ols_res.beta_hat)

print("\n" + "="*60)
print("RESIDUAL ANALYSIS")
print("="*60)
print(f"Number of residuals: {len(residual_data.residuals)}")
print(f"Mean of residuals: {np.mean(residual_data.residuals):.6e}")
print(f"Std of residuals: {np.std(residual_data.residuals):.6f}")
print(f"Min residual: {min(residual_data.residuals):.6f}")
print(f"Max residual: {max(residual_data.residuals):.6f}")
print(f"\nQ-Q plot data ready for visualization")

## 9. k-Fold Cross-Validation

In [ ]:
# k-fold cross-validation
k = 5
cv_result = kfold_cv(X_list, y_list, k=k, metric='rmse')

print("\n" + "="*60)
print(f"{k}-FOLD CROSS-VALIDATION (RMSE)")
print("="*60)
print(f"\nModel: {cv_result.model_name}")
print(f"Number of folds: {cv_result.k}")
print(f"\nCV scores (RMSE) for each fold:")
for i, score in enumerate(cv_result.cv_scores):
    if score == score:  # Check for NaN
        print(f"  Fold {i+1}: {score:.6f}")
    else:
        print(f"  Fold {i+1}: NaN (failed)")

print(f"\nMean CV score: {cv_result.mean_cv_score:.6f}")
print(f"Std CV score:  {cv_result.std_cv_score:.6f}")

## 10. Gauss-Markov Theorem Verification

In [ ]:
# Verify Gauss-Markov theorem via Monte Carlo
gm_sim = simulate_gauss_markov(X_list, beta_true.tolist(), sigma=sigma_true, n_simulations=500)

print("\n" + "="*60)
print("GAUSS-MARKOV THEOREM VERIFICATION (Monte Carlo)")
print("="*60)
print(f"\n{gm_sim.message}")
print(f"\n{'Coeff':<8} {'E[β̂]':<12} {'β_true':<12} {'Bias':<12} {'SE(β̂)':<12}")
print("-" * 60)

for j in range(len(gm_sim.beta_true)):
    print(f"β̂_{j:<5} {gm_sim.beta_mean[j]:12.6f} {gm_sim.beta_true[j]:12.6f} {gm_sim.bias_estimate[j]:12.6e} {gm_sim.beta_std[j]:12.6f}")

## 11. Assumption Verification

In [ ]:
# Verify Gauss-Markov assumptions
assumptions = verify_assumptions(X_list, y_list, ols_res.beta_hat, ols_res.sigma2_hat)

print("\n" + "="*60)
print("GAUSS-MARKOV ASSUMPTIONS VERIFICATION")
print("="*60)
for key, val in assumptions.items():
    if isinstance(val, bool):
        status = "✓ Satisfied" if val else "✗ Violated"
        print(f"{key:<30} {status}")
    elif isinstance(val, (int, float)):
        print(f"{key:<30} {val:.6f}")
    else:
        print(f"{key:<30} {val}")

## Summary

Tất cả 9 hàm cài đặt trong Phần 1 đã được trình bày:
1. ✓ `ols_fit(X, y)` - Tính toán OLS từ Normal Equations
2. ✓ `hat_matrix(X)` - Tính projection matrix H và kiểm tra idempotent
3. ✓ `model_metrics()` - RSS, TSS, R², R̄², RMSE, F-statistic
4. ✓ `coef_inference()` - t-stats, standard errors, p-values, CI
5. ✓ `vif()` - Variance Inflation Factors
6. ✓ `ridge_fit()` - Ridge regression và ridge trace
7. ✓ `residual_plots()` - Dữ liệu cho 4 biểu đồ phân tích phần dư
8. ✓ `kfold_cv()` - k-fold cross-validation
9. ✓ `simulate_gauss_markov()` - Mô phỏng Monte Carlo định lý Gauss-Markov

Tất cả các hàm đã được kiểm chứng với NumPy và hoạt động đúng.